# 🧹 Automated Seed Cleaner

Welcome to the Seed Cleaner! When you collect physical data using the smart pen, it's normal to occasionally drop the pen, draw a messy character, or pause too long. 

Because the augmentation script multiplies each of your physical examples by 1,000 to train the network, **one bad seed creates 1,000 bad training samples!** This notebook uses statistical Z-scores to automatically weed out anomalies.

## 1. Imports and Setup
First, we'll load the data science libraries we need for time-series analysis.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## 2. Configuration
Define what labels you're currently working with and how strict you want the anomaly detection to be.

In [ ]:
# The target labels you want to clean
LABELS = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
SEED_DIR = '../data/seed'

# Z-Score Threshold (Standard Deviations from the Mean)
# 2.0 = Strict (deletes anything slightly weird)
# 3.0 = Generous (only deletes utterly crazy strokes)
Z_THRESH = 2.0  

## 3. Analysis Pipeline
The core algorithm. This runs through every `csv` file and computes:
- **Stroke Length:** How long the drawing took.
- **Peak Gyro:** The absolute fastest angular velocity of the stroke.
- **Total Energy:** The sum magnitude of the movement.

It then compares all samples of a single label against the bell curve average.

In [ ]:
def extract_features(file_path):
    """Reads a CSV and returns core metrics for anomaly detection."""
    df = pd.read_csv(file_path)
    gyro_mag = np.sqrt(df['gx']**2 + df['gy']**2 + df['gz']**2)
    return {
        'file': file_path,
        'length': len(df),
        'peak_gyro': gyro_mag.max(),
        'energy': gyro_mag.sum(),
        'df': df
    }

## 4. Run Diagnosis
Run this block to inspect your seeds. This will **NOT** delete anything yet. It just prints a visual overlay so you can visually verify if the algorithm was right to flag an anomaly.

In [ ]:
bad_files_to_delete = []

for label in LABELS:
    path = os.path.join(SEED_DIR, label)
    if not os.path.exists(path): 
        continue
        
    files = sorted(glob.glob(os.path.join(path, "*.csv")))
    if len(files) < 5: 
        print(f"[!] Skipping '{label}' - needs at least 5 samples to build a statistical bell curve.")
        continue
        
    # Load all metrics
    metrics = [extract_features(f) for f in files]
    sdf = pd.DataFrame(metrics)
    
    # Calculate standard deviations from the average
    z_len = np.abs(stats.zscore(sdf['length']))
    z_peak = np.abs(stats.zscore(sdf['peak_gyro']))
    z_energy = np.abs(stats.zscore(sdf['energy']))
    
    # Detect outliers across ANY of the three metrics
    outliers = (z_len > Z_THRESH) | (z_peak > Z_THRESH) | (z_energy > Z_THRESH)
    
    if not outliers.any():
        print(f"✅ Digit {label} is completely clean! No outliers detected.")
        continue
        
    # --- Rendering Comparisons ---
    print(f"\n❌ {outliers.sum()} Anomalies detected for digit '{label}'")
    
    good_count = sum(~outliers)
    bad_count = sum(outliers)
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    
    # Plot 1: Normal Cloud
    axes[0].set_title(f"GOOD Samples ({good_count})")
    axes[0].set_ylabel("Gyroscope Dynamics (deg/s)")
    for i, is_out in enumerate(outliers):
        if not is_out:
            df = metrics[i]['df']
            axes[0].plot(df['gx'], 'r', alpha=0.25)
            axes[0].plot(df['gy'], 'g', alpha=0.25)
            axes[0].plot(df['gz'], 'b', alpha=0.25)
            
    # Plot 2: Anomalies
    axes[1].set_title(f"FLAGGED Anomalies ({bad_count})")
    for i, is_out in enumerate(outliers):
        if is_out:
            bad_fname = metrics[i]['file']
            bad_files_to_delete.append(bad_fname)
            
            print(f"  → Flagged: {os.path.basename(bad_fname)}  (Z-len: {z_len[i]:.1f}, Z-peak: {z_peak[i]:.1f}, Z-erg: {z_energy[i]:.1f})")
            
            df = metrics[i]['df']
            axes[1].plot(df['gx'], 'r', alpha=0.9)
            axes[1].plot(df['gy'], 'g', alpha=0.9)
            axes[1].plot(df['gz'], 'b', alpha=0.9)
            
    axes[0].sharey(axes[1])
    plt.tight_layout()
    plt.show()

## 5. Purge Anomalies
If you agree with the algorithm's flags above, run this final block to delete the corrupted CSVs from the disk.

In [ ]:
if not bad_files_to_delete:
    print("There are no bad files waiting to be deleted! You're good to go.")
else:
    print(f"Executing purge of {len(bad_files_to_delete)} files...")
    for f in bad_files_to_delete:
        if os.path.exists(f):
            os.remove(f)
            print(f"  [DELETED] {f}")
            
    print("\nDone! Dataset successfully sterilized.")
    
    # Clear the queue so we don't accidentally double-run it
    bad_files_to_delete = []